# driving-risk-metrics — train the comparison on Colab

Trains four segmentation architectures on CamVid under **one identical budget**,
then scores them with risk-weighted, distance-stratified metrics.

The point of the identical budget is methodological. The source notebooks this
repository grew out of tuned each model separately and then compared the numbers,
which makes the comparison uninterpretable — you cannot tell an architecture
difference from a tuning difference. Here every model gets the same epochs,
batch size, optimiser, schedule, augmentation and seed.

**Runtime → Change runtime type → GPU** before running. An L4 or A100 will do all
four models comfortably; a T4 works but is slower.

---


## 1. Environment


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 2. Get the code

Clone the repo, or upload it if you are working from a local copy.


In [ ]:
REPO = 'https://github.com/kuotunyu/driving-risk-metrics.git'

import os, pathlib
if not pathlib.Path('driving-risk-metrics').exists():
    !git clone -q $REPO
%cd driving-risk-metrics
!pip install -q -e '.[train,report]'


## 3. Get CamVid

CamVid is **not** redistributed by this repository. Fetch it yourself and point
`CAMVID_ROOT` at the directory holding `train.csv` and `val.csv`.

The cell below expects a copy in your Drive; adapt the path, or upload the
extracted folder directly to the Colab filesystem.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CAMVID_ROOT = '/content/drive/MyDrive/CamVid'  # <- edit this

import pathlib
root = pathlib.Path(CAMVID_ROOT)
assert (root / 'train.csv').exists(), f'no train.csv under {root}'
print('CamVid found at', root)


### Freeze the split and measure the class prior

This writes a manifest with a SHA-256 per file, so any later run either sees the
same bytes or fails loudly. It also reproduces the class-imbalance measurement
the whole package is built around.


In [ ]:
!python scripts/analyse_dataset.py --root "$CAMVID_ROOT"


## 4. Validate the pipeline before spending GPU time

Synthetic fixtures exercise the entire evaluation and reporting chain without a
model. If anything downstream is broken, it is much cheaper to find out here.

These are **not** model results and are banner-marked as such in the report.


In [ ]:
!python scripts/evaluate.py --root "$CAMVID_ROOT" --split val --synthetic
!python scripts/report.py --out reports/report_synthetic.html


## 5. Train

Four architectures, one budget. Three of them (FCN8s, DeepLabV3, SETR) correspond
to models in the source notebooks, so their results under this protocol can be set
against the originals; SegFormer is a modern baseline.

Roughly 20–40 min each on an L4 at 40 epochs. Start with `EPOCHS = 5` to confirm
the loop runs before committing to the full budget.


In [ ]:
EPOCHS = 40
BATCH  = 8
SEED   = 0
MODELS = ['fcn8s', 'deeplabv3_resnet50', 'setr_pup', 'segformer_b0']

for m in MODELS:
    print(f'
{"=" * 70}
{m}
{"=" * 70}')
    !python scripts/train.py --root "$CAMVID_ROOT" --model $m \
        --epochs $EPOCHS --batch-size $BATCH --seed $SEED --num-workers 2


### Optional: does class weighting fix the blindness?

Plain cross-entropy is the condition under which the observed models never
recovered a pedestrian. Re-running one model with inverse-square-root class
weights turns the obvious remedy into a measurement rather than an assumption.

Note that this changes the *training* objective, not the evaluation — the metrics
are identical for both runs, which is what makes them comparable.


In [ ]:
!python scripts/train.py --root "$CAMVID_ROOT" --model deeplabv3_resnet50 \
    --epochs $EPOCHS --batch-size $BATCH --seed $SEED \
    --class-weights inverse-sqrt --out runs/weighted


## 6. Evaluate and report

One traversal per model produces every metric, so the numbers in the report
cannot drift apart from each other.


In [ ]:
preds = ' '.join(f'{m}=predictions/{m}' for m in MODELS)
!python scripts/evaluate.py --root "$CAMVID_ROOT" --split val --predictions $preds
!python scripts/report.py


In [ ]:
from IPython.display import HTML, display
display(HTML(open('reports/report.html', encoding='utf-8').read()))


## 7. Take the results home

The report is a single self-contained HTML file; the JSON alongside it is the
machine-readable evidence every number in the page was rendered from.


In [ ]:
import shutil, pathlib
shutil.make_archive('/content/drivemetrics_results', 'zip', 'reports')

from google.colab import files
files.download('/content/drivemetrics_results.zip')


---

### What to look for in the report

- **`classes` beside every mIoU.** A three-class mean and an eleven-class mean are
  different quantities. Conflating them is how the source material came to report
  four incomparable numbers as a ranking.
- **`protocol gap`.** How far the per-image `nanmean` aggregation drifts from the
  dataset-level one. On the source material that gap exceeded the spread between
  the architectures being compared.
- **Blind-spot rate.** How often a pedestrian that was present was not recovered
  at all. A model can carry a respectable mIoU with a blind-spot rate of 100%.
- **The harm sweep.** Whether the ranking survives a different opinion about how
  much a missed person costs. If it does not, say so — that is the finding.

This is a research and teaching artefact. It is not a safety case and does not
constitute validation of any vehicle.
